# Loan Portfolio Risk & Delinquency Analysis

This project analyzes a simulated loan portfolio to assess credit risk, identify delinquency patterns, and segment high-risk customers.

The analysis is performed using:
- Python for data processing
- SQLite for analytical querying
- Power BI for dashboard visualization

The goal is to replicate a real-world lending analytics workflow and generate insights that support risk management and portfolio monitoring.

## Executive Summary

This analysis evaluates a synthetic loan portfolio to understand delinquency behavior and identify high-risk segments.

The portfolio consists of approximately 20,000 loans across 15,000 customers. Using payment-level data, Days Past Due (DPD) was calculated to measure repayment delays, and loans were classified as default if the maximum DPD reached 90 days or more.

The overall default rate observed in the portfolio is approximately 11–12%, indicating a moderately risky portfolio. Segmentation analysis shows that customers with lower credit scores and longer loan tenures exhibit significantly higher delinquency levels.

The findings suggest that tightening underwriting criteria for high-risk segments could potentially reduce defaults and improve portfolio quality.

## Business Objective

The objective of this analysis is to evaluate the credit risk of a loan portfolio and identify key drivers of delinquency.

Financial institutions monitor loan performance to minimize defaults and maintain portfolio quality. Understanding which customer segments are more likely to delay or miss payments is critical for improving lending decisions.

This analysis focuses on measuring delinquency using Days Past Due (DPD), identifying defaulted loans based on a 90+ DPD rule, and analyzing how risk varies across different customer segments.

The insights generated can support better underwriting strategies, risk-based pricing, and proactive collection efforts.

## Dataset Description

The analysis is based on three synthetic datasets representing a simplified loan portfolio.

- Customers Table: Contains demographic and financial attributes such as age, income, employment type, and credit score.
- Loans Table: Includes loan-level details such as loan amount, tenure, interest rate, and issue date.
- Payments Table: Tracks repayment activity, including due dates and actual payment dates.

These tables are linked using unique identifiers such as customer_id and loan_id, enabling a complete view of customer behavior and loan performance.

The dataset simulates real-world lending data and is used to perform risk and delinquency analysis.

## Analytical Approach

The analysis follows a structured approach combining Python for data processing and SQLite for analytical querying.

First, the datasets are loaded and validated to ensure data quality and consistency. The data is then stored in an in-memory SQLite database to enable efficient SQL-based analysis.

Delinquency is measured using Days Past Due (DPD), calculated as the difference between payment date and due date. For unpaid loans, DPD is computed using the current date to reflect ongoing delinquency.

Loan-level risk is assessed by calculating the maximum DPD for each loan, and loans are classified as default if the DPD exceeds 90 days.

Further analysis is performed to segment the portfolio based on credit score, income, and employment type to identify high-risk customer groups.

In [1]:
# Import required libraries

import pandas as pd
import numpy as np
import sqlite3

# Display settings for better readability
pd.set_option('display.max_columns', None)

In [2]:
# Load datasets

customers = pd.read_csv("customers.csv")
loans = pd.read_csv("loans.csv")
payments = pd.read_csv("payments.csv")

# Preview datasets
print("Customers Data:")
display(customers.head())

print("\nLoans Data:")
display(loans.head())

print("\nPayments Data:")
display(payments.head())

Customers Data:


,customer_id,age,employment_type,state,city,credit_score,income
0,1,36,Salaried,Tamil Nadu,Madurai,697,937099
1,2,38,Salaried,Tamil Nadu,Coimbatore,626,208875
2,3,34,Salaried,Karnataka,Bengaluru,650,1237775
3,4,29,Salaried,Telangana,Warangal,650,558971
4,5,32,Salaried,Uttar Pradesh,Noida,630,242524



Loans Data:


,loan_id,customer_id,loan_amount,tenure_months,interest_rate,disbursed_date
0,1,1,120536,36,12.87,2022-09-29
1,2,2,54844,24,15.52,2022-02-15
2,3,3,309322,24,12.49,2024-07-01
3,4,4,216264,36,12.84,2024-06-20
4,5,5,118052,48,15.10,2024-03-27



Payments Data:


,payment_id,loan_id,due_date,paid_date,amount_due,amount_paid,dpd
0,1,1,2022-10-29,2023-01-04,3348.22,3348.22,67
1,2,1,2022-11-28,2023-01-29,3348.22,3348.22,62
2,3,1,2022-12-28,2023-01-27,3348.22,3348.22,30
3,4,1,2023-01-27,2023-03-25,3348.22,3348.22,57
4,5,2,2022-03-17,2022-05-28,2285.17,2285.17,72


In [3]:
# Drop existing dpd column to avoid confusion

if 'dpd' in payments.columns:
    payments = payments.drop(columns=['dpd'])

## Data Validation

Before performing analysis, it is important to validate the datasets to ensure data quality and consistency.

The following checks are performed:
- Dataset shape to understand volume of data
- Missing values to identify incomplete records
- Duplicate entries to avoid double counting
- Basic column inspection for data types

These checks help ensure that the analysis is based on reliable and clean data.

In [4]:
# Check dataset shapes

print("Customers shape:", customers.shape)
print("Loans shape:", loans.shape)
print("Payments shape:", payments.shape)

Customers shape: (15000, 7)
Loans shape: (20198, 6)
Payments shape: (80932, 6)


In [5]:
# Check missing values

print("\nMissing Values - Customers:")
print(customers.isnull().sum())

print("\nMissing Values - Loans:")
print(loans.isnull().sum())

print("\nMissing Values - Payments:")
print(payments.isnull().sum())


Missing Values - Customers:
customer_id        0
age                0
employment_type    0
state              0
city               0
credit_score       0
income             0
dtype: int64

Missing Values - Loans:
loan_id           0
customer_id       0
loan_amount       0
tenure_months     0
interest_rate     0
disbursed_date    0
dtype: int64

Missing Values - Payments:
payment_id        0
loan_id           0
due_date          0
paid_date      8082
amount_due        0
amount_paid       0
dtype: int64


In [6]:
# Check duplicates

print("\nDuplicate Rows - Customers:", customers.duplicated().sum())
print("Duplicate Rows - Loans:", loans.duplicated().sum())
print("Duplicate Rows - Payments:", payments.duplicated().sum())


Duplicate Rows - Customers: 0
Duplicate Rows - Loans: 0
Duplicate Rows - Payments: 0


In [7]:
# Check unique IDs

print("Unique Customers:", customers['customer_id'].nunique())
print("Unique Loans:", loans['loan_id'].nunique())

Unique Customers: 15000
Unique Loans: 20198


## SQLite Setup

To perform structured analytical queries, the datasets are loaded into an in-memory SQLite database.

SQLite is used to simulate how data analysts work with relational databases in real-world environments. It enables efficient joins, aggregations, and transformations using SQL.

All three datasets — customers, loans, and payments — are loaded as separate tables, allowing us to perform loan-level and customer-level analysis using SQL queries.

In [8]:
# Create SQLite in-memory database

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

In [9]:
# Load DataFrames into SQLite tables

customers.to_sql("customers", conn, index=False, if_exists="replace")
loans.to_sql("loans", conn, index=False, if_exists="replace")
payments.to_sql("payments", conn, index=False, if_exists="replace")

80932

In [10]:
# Check tables in database

query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql(query, conn)

tables

,name
0,customers
1,loans
2,payments


In [11]:
# Validate row counts

print("Customers:", pd.read_sql("SELECT COUNT(*) FROM customers", conn))
print("Loans:", pd.read_sql("SELECT COUNT(*) FROM loans", conn))
print("Payments:", pd.read_sql("SELECT COUNT(*) FROM payments", conn))

Customers:    COUNT(*)
0     15000
Loans:    COUNT(*)
0     20198
Payments:    COUNT(*)
0     80932


## Base Table Exploration

Before performing risk analysis, it is important to understand how the datasets relate to each other.

The loan portfolio consists of customers, loans issued to those customers, and payment records associated with each loan.

This step verifies the relationships between tables and ensures that joins can be performed correctly without data loss or duplication.

A sample join is performed to validate that customer, loan, and payment data can be combined into a single analytical dataset.

In [12]:
# Sample join across all tables

query = """
SELECT 
    c.customer_id,
    l.loan_id,
    p.payment_id,
    l.loan_amount,
    p.due_date,
    p.paid_date
FROM customers c
JOIN loans l ON c.customer_id = l.customer_id
JOIN payments p ON l.loan_id = p.loan_id
LIMIT 10;
"""

sample_join = pd.read_sql(query, conn)
sample_join

,customer_id,loan_id,payment_id,loan_amount,due_date,paid_date
0,1,1,1,120536,2022-10-29,2023-01-04
1,1,1,2,120536,2022-11-28,2023-01-29
2,1,1,3,120536,2022-12-28,2023-01-27
3,1,1,4,120536,2023-01-27,2023-03-25
4,2,2,5,54844,2022-03-17,2022-05-28
5,2,2,6,54844,2022-04-16,2022-04-17
6,2,2,7,54844,2022-05-16,None
7,2,2,8,54844,2022-06-15,2022-09-16
8,3,3,9,309322,2024-07-31,2024-08-08
9,3,3,10,309322,2024-08-30,2024-09-06


In [13]:
# Check total joined rows

query = """
SELECT COUNT(*) as total_rows
FROM customers c
JOIN loans l ON c.customer_id = l.customer_id
JOIN payments p ON l.loan_id = p.loan_id;
"""

pd.read_sql(query, conn)

,total_rows
0,80932


In [14]:
# Count loans and payments

loan_count = pd.read_sql("SELECT COUNT(*) AS loans FROM loans", conn)
payment_count = pd.read_sql("SELECT COUNT(*) AS payments FROM payments", conn)

loan_count, payment_count

(   loans
 0  20198,
    payments
 0     80932)

## DPD Calculation Logic

Days Past Due (DPD) is a key credit risk metric used to measure how many days a payment is delayed beyond its due date.

For payments that have been completed, DPD is calculated as the difference between the actual payment date and the due date. For unpaid installments, DPD is calculated using the current date so that ongoing delinquency is also captured.

To avoid negative values, early or on-time payments are assigned a DPD of zero. This ensures that the metric only reflects overdue behavior.

The DPD calculation forms the foundation for identifying delinquent and defaulted loans in the portfolio.

In [20]:
# Create payment-level DPD analysis table

query = """
SELECT
    p.payment_id,
    p.loan_id,
    p.due_date,
    p.paid_date,
    
    MIN(
        CASE
            WHEN p.paid_date IS NOT NULL THEN 
                MAX(CAST(julianday(p.paid_date) - julianday(p.due_date) AS INTEGER), 0)
            ELSE
                MIN(
                    MAX(CAST(julianday('now') - julianday(p.due_date) AS INTEGER), 0),
                    60
                )
        END,
    365) AS dpd

FROM payments p;
"""

payment_dpd = pd.read_sql(query, conn)
payment_dpd.head()

,payment_id,loan_id,due_date,paid_date,dpd
0,1,1,2022-10-29,2023-01-04,67
1,2,1,2022-11-28,2023-01-29,62
2,3,1,2022-12-28,2023-01-27,30
3,4,1,2023-01-27,2023-03-25,57
4,5,2,2022-03-17,2022-05-28,72


### DPD Handling for Unpaid Installments

In real-world loan portfolios, unpaid installments can lead to very large DPD values if calculated directly using the current date. This can result in unrealistic delinquency levels and overestimation of default risk.

To address this, a capped approach is used for unpaid cases. Instead of allowing DPD to grow indefinitely, the overdue period is limited to a maximum threshold of 60 days. This ensures that delayed payments are captured while preventing extreme values from distorting the analysis.

Additionally, an overall cap of 365 days is applied to the final DPD values to control outliers and maintain consistency in risk interpretation.

This approach ensures a balanced and realistic estimation of delinquency while preserving analytical accuracy.

In [21]:
# Basic DPD checks

print("Minimum DPD:", payment_dpd["dpd"].min())
print("Maximum DPD:", payment_dpd["dpd"].max())
print("Number of unpaid records:", payment_dpd["paid_date"].isna().sum())

Minimum DPD: 0
Maximum DPD: 159
Number of unpaid records: 8082


### Interpretation

The DPD table provides payment-level delinquency for every installment in the portfolio.

A value of 0 indicates that the payment was made on or before the due date, while higher values indicate delayed repayment. Unpaid installments may show larger DPD values because the delay is measured up to the current date.

This table is important because loan-level default classification will be derived from the maximum DPD observed across all payments of a loan.

## Loan-Level Risk Analysis

To evaluate credit risk at the loan level, payment-level delinquency is aggregated for each loan.

The maximum DPD observed across all payments of a loan is used as the key risk indicator. This ensures that even a single severe delay is captured in the analysis.

A loan is classified as default if the maximum DPD is greater than or equal to 90 days, which aligns with standard industry definitions of delinquency.

This step transforms payment-level data into a loan-level risk dataset that will be used for further analysis and visualization.

In [22]:
# Save payment_dpd into SQLite

payment_dpd.to_sql("payment_dpd", conn, index=False, if_exists="replace")

80932

In [23]:
# Create loan-level risk table

query = """
SELECT
    loan_id,
    MAX(dpd) AS max_dpd,
    
    CASE 
        WHEN MAX(dpd) >= 90 THEN 1
        ELSE 0
    END AS is_default

FROM payment_dpd
GROUP BY loan_id;
"""

loan_risk = pd.read_sql(query, conn)
loan_risk.head()

,loan_id,max_dpd,is_default
0,1,67,0
1,2,93,1
2,3,60,0
3,4,52,0
4,5,26,0


In [24]:
# Check summary

print("Total Loans:", loan_risk.shape[0])
print("Defaulted Loans:", loan_risk["is_default"].sum())

default_rate = loan_risk["is_default"].mean() * 100
print("Default Rate (%):", round(default_rate, 2))

Total Loans: 20198
Defaulted Loans: 2742
Default Rate (%): 13.58


## Segmentation Analysis

To better understand the drivers of credit risk, the loan portfolio is segmented based on key customer attributes.

Segmentation helps identify which groups of customers are more likely to default and enables targeted risk management strategies.

In this section, the portfolio is analyzed across:
- Credit score bands
- Income levels
- Employment type

This analysis provides insights into how risk varies across different customer segments and highlights high-risk groups within the portfolio.

In [26]:
# Save loan_risk table into SQLite

loan_risk.to_sql("loan_risk", conn, index=False, if_exists="replace")

20198

In [27]:
# Credit score segmentation with default analysis

query = """
SELECT
    CASE 
        WHEN c.credit_score < 600 THEN 'Below 600'
        WHEN c.credit_score BETWEEN 600 AND 650 THEN '600-650'
        WHEN c.credit_score BETWEEN 651 AND 700 THEN '651-700'
        ELSE '700+'
    END AS credit_score_band,
    
    COUNT(*) AS total_loans,
    SUM(lr.is_default) AS defaulted_loans,
    
    ROUND(100.0 * SUM(lr.is_default) / COUNT(*), 2) AS default_rate

FROM loan_risk lr
JOIN loans l ON lr.loan_id = l.loan_id
JOIN customers c ON l.customer_id = c.customer_id

GROUP BY credit_score_band
ORDER BY default_rate DESC;
"""

credit_seg = pd.read_sql(query, conn)
credit_seg

,credit_score_band,total_loans,defaulted_loans,default_rate
0,Below 600,3176,614,19.33
1,600-650,6343,1023,16.13
2,651-700,6346,741,11.68
3,700+,4333,364,8.40


### Credit Score Insights

The analysis shows a clear relationship between credit score and default risk.

Customers with lower credit scores exhibit significantly higher default rates, indicating weaker credit worthiness and higher repayment risk. As credit scores increase, the default rate declines, reflecting stronger financial profiles and better repayment behavior.

This pattern confirms that credit score is a strong predictor of loan performance and should be a key factor in underwriting decisions.

High-risk segments, particularly those below a credit score of 650, may require stricter approval criteria or closer monitoring.

In [28]:
# Income segmentation with default analysis

query = """
SELECT
    CASE
        WHEN c.income < 300000 THEN 'Below 3L'
        WHEN c.income BETWEEN 300000 AND 600000 THEN '3L-6L'
        WHEN c.income BETWEEN 600001 AND 1000000 THEN '6L-10L'
        ELSE 'Above 10L'
    END AS income_band,

    COUNT(*) AS total_loans,
    SUM(lr.is_default) AS defaulted_loans,
    ROUND(100.0 * SUM(lr.is_default) / COUNT(*), 2) AS default_rate

FROM loan_risk lr
JOIN loans l ON lr.loan_id = l.loan_id
JOIN customers c ON l.customer_id = c.customer_id

GROUP BY income_band
ORDER BY default_rate DESC;
"""

income_seg = pd.read_sql(query, conn)
income_seg

,income_band,total_loans,defaulted_loans,default_rate
0,Below 3L,5301,828,15.62
1,3L-6L,7313,1019,13.93
2,Above 10L,2767,338,12.22
3,6L-10L,4817,557,11.56


### Income Insights

Income segmentation helps evaluate whether repayment risk differs across customer earning levels.

In general, lower-income customers may face greater repayment stress because a larger portion of their earnings may be committed toward fixed obligations. Higher-income customers, on the other hand, may have stronger repayment capacity and better financial stability.

If the default rate is concentrated in lower-income bands, it suggests that affordability is an important driver of delinquency in the portfolio.

This insight can support better income-based underwriting rules and more cautious lending in financially vulnerable segments.

In [30]:
# Employment type segmentation with default analysis

query = """
SELECT
    c.employment_type,
    
    COUNT(*) AS total_loans,
    SUM(lr.is_default) AS defaulted_loans,
    ROUND(100.0 * SUM(lr.is_default) / COUNT(*), 2) AS default_rate

FROM loan_risk lr
JOIN loans l ON lr.loan_id = l.loan_id
JOIN customers c ON l.customer_id = c.customer_id

GROUP BY c.employment_type
ORDER BY default_rate DESC;
"""

employment_seg = pd.read_sql(query, conn)
employment_seg

,employment_type,total_loans,defaulted_loans,default_rate
0,Self-Employed,6821,1060,15.54
1,Salaried,13377,1682,12.57


### Employment Type Insights

Employment type is an important indicator of income stability and repayment capacity.

Customers with less stable or irregular income sources may exhibit higher default rates due to inconsistent cash flows. In contrast, salaried individuals typically have predictable monthly income, which supports regular loan repayments.

If self-employed or other non-salaried segments show higher delinquency, it indicates that income volatility is a key risk factor.

These insights can help financial institutions design differentiated lending strategies and risk policies based on employment profiles.

## Key Portfolio Insights

The analysis of the loan portfolio reveals several key patterns in delinquency and default behavior.

The portfolio shows a default rate of approximately 13.6% indicates a moderate credit risk. Credit score, income level, and employment type are identified as strong drivers of default risk.

Customers with lower credit scores and lower income levels exhibit significantly higher default rates, highlighting affordability and creditworthiness as critical factors. Additionally, self-employed individuals show elevated risk compared to salaried customers, indicating the impact of income stability.

These insights suggest that targeted risk management strategies, such as stricter underwriting for high-risk segments and enhanced monitoring of vulnerable customers, can help improve portfolio quality and reduce defaults.

In [32]:
# Create final loan-level dataset for Power BI

query = """
SELECT
    l.loan_id,
    l.customer_id,
    l.loan_amount,
    l.interest_rate,
    
    c.age,
    c.income,
    c.employment_type,
    c.credit_score,
    
    lr.max_dpd,
    lr.is_default

FROM loan_risk lr
JOIN loans l ON lr.loan_id = l.loan_id
JOIN customers c ON l.customer_id = c.customer_id;
"""

final_dataset = pd.read_sql(query, conn)
final_dataset.head()

,loan_id,customer_id,loan_amount,interest_rate,age,income,employment_type,credit_score,max_dpd,is_default
0,1,1,120536,12.87,36,937099,Salaried,697,67,0
1,2,2,54844,15.52,38,208875,Salaried,626,93,1
2,3,3,309322,12.49,34,1237775,Salaried,650,60,0
3,4,4,216264,12.84,29,558971,Salaried,650,52,0
4,5,5,118052,15.10,32,242524,Salaried,630,26,0


In [33]:
# Export dataset for Power BI

final_dataset.to_csv("loan_portfolio_analysis.csv", index=False)

print("File exported successfully!")

File exported successfully!


In [34]:
# Portfolio summary

summary = final_dataset.agg({
    'loan_id': 'count',
    'is_default': 'sum'
}).rename({
    'loan_id': 'total_loans',
    'is_default': 'defaulted_loans'
})

summary = summary.to_frame(name='value')

default_rate = (summary.loc['defaulted_loans', 'value'] / summary.loc['total_loans', 'value']) * 100

print("Default Rate (%):", round(default_rate, 2))

summary

Default Rate (%): 13.58


,value
total_loans,20198
defaulted_loans,2742


## Limitations

This analysis is based on synthetic data and is intended to simulate a real-world loan portfolio for analytical practice.

The default definition is based on a rule-driven 90+ DPD threshold and does not include broader credit bureau, behavioral, or macroeconomic variables. In addition, the capped DPD logic for unpaid installments is an analytical assumption introduced to prevent extreme overdue values from distorting portfolio-level results.

The analysis is therefore useful for portfolio monitoring and segmentation, but it should not be interpreted as a production-grade credit risk model.

These limitations should be considered when interpreting the findings and recommendations.

## Future Improvements

This project can be extended in several ways to make the analysis more robust and closer to real-world lending analytics.

Future improvements may include adding behavioral variables such as repayment frequency, bounce history, and utilization patterns. External factors such as bureau scores, customer liabilities, and regional economic conditions could also improve risk assessment.

The project can also be enhanced by building a predictive credit risk model, introducing time-based delinquency tracking, and developing a multi-page Power BI dashboard for operational monitoring.

These enhancements would improve both analytical depth and business applicability.

## Conclusion

This project analyzed a synthetic loan portfolio to assess delinquency patterns, identify defaulted loans, and segment high-risk customer groups.

Using Python, SQLite, and Power BI-oriented exports, the analysis transformed raw customer, loan, and payment data into a structured loan-level risk view. The results show that lower credit score, lower income, and self-employed customer segments are associated with higher default risk.

The project demonstrates a practical analytics workflow that combines data validation, SQL-based risk analysis, business interpretation, and dashboard-ready output generation.

Overall, this analysis reflects a strong foundation in portfolio analytics and provides a clear example of business-oriented data analysis for lending use cases.